[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/08_data_cleaning/08_7_Dates_Times.ipynb)

# 08.7: Dates and Times

Almost every real-world dataset has a date or timestamp column. And almost every CSV file stores those dates as strings. pandas reads string columns as `object` dtype, which means you cannot sort dates reliably across all formats, filter rows by date range, or subtract two dates to find a duration.

This notebook covers three things: parsing string dates into `datetime64` values with `pd.to_datetime()`, extracting date components with the `.dt` accessor, and computing durations by subtracting datetime columns. Let's look at the problem.

In [1]:
import pandas as pd
import seaborn as sns

## Why string dates are broken

Suppose you have a column of dates stored as strings. Sorting them looks fine at first glance, but the sort is alphabetical, not chronological. Alphabetical sort works for ISO 8601 format (`"YYYY-MM-DD"`) by coincidence, because the year comes first. It breaks for every other common format.

In [2]:
# Three common date formats for the same set of dates
dates_iso = pd.Series(["2023-03-15", "2023-01-02", "2023-11-30", "2023-06-08"])
dates_us  = pd.Series(["03/15/2023", "01/02/2023", "11/30/2023", "06/08/2023"])
dates_text = pd.Series(["March 15, 2023", "January 2, 2023", "November 30, 2023", "June 8, 2023"])

print("ISO format sorted:  ", sorted(dates_iso.tolist()))
print("US format sorted:   ", sorted(dates_us.tolist()))
print("Text format sorted: ", sorted(dates_text.tolist()))

ISO format sorted:   ['2023-01-02', '2023-03-15', '2023-06-08', '2023-11-30']
US format sorted:    ['01/02/2023', '03/15/2023', '06/08/2023', '11/30/2023']
Text format sorted:  ['January 2, 2023', 'June 8, 2023', 'March 15, 2023', 'November 30, 2023']


ISO format sorts correctly by coincidence. US format (`MM/DD/YYYY`) sorts `01/02/2023` before `03/15/2023` alphabetically, which happens to be correct here, but only because the month numbers happen to sort in the right order for this particular dataset. The text format sorts `"January"` before `"June"` before `"March"` before `"November"` alphabetically, which is wrong: January and March are in the right order, but June comes before November and both should come after March.

The fix is a single call: `pd.to_datetime()`.

## Parsing dates with `pd.to_datetime()`

`pd.to_datetime()` converts a Series of strings into `datetime64` values. For ISO format dates it works with no arguments beyond the data.

In [3]:
parsed = pd.to_datetime(dates_iso)
print("dtype:", parsed.dtype)
print(parsed.sort_values().tolist())

dtype: datetime64[us]
[Timestamp('2023-01-02 00:00:00'), Timestamp('2023-03-15 00:00:00'), Timestamp('2023-06-08 00:00:00'), Timestamp('2023-11-30 00:00:00')]


The dtype is now `datetime64[ns]` (nanosecond precision). Sorting is chronological, not alphabetical.

For formats that are not ISO 8601, pandas needs a hint about how the string is structured. The `format=` argument takes a format code string where `%m` means month, `%d` means day, and `%Y` means four-digit year. Providing the format is faster and avoids ambiguity.

In [4]:
# US format: month/day/year
parsed_us = pd.to_datetime(dates_us, format="%m/%d/%Y")
print("US format parsed and sorted:")
print(parsed_us.sort_values().tolist())

# Text format
parsed_text = pd.to_datetime(dates_text, format="%B %d, %Y")
print("\nText format parsed and sorted:")
print(parsed_text.sort_values().tolist())

US format parsed and sorted:
[Timestamp('2023-01-02 00:00:00'), Timestamp('2023-03-15 00:00:00'), Timestamp('2023-06-08 00:00:00'), Timestamp('2023-11-30 00:00:00')]

Text format parsed and sorted:
[Timestamp('2023-01-02 00:00:00'), Timestamp('2023-03-15 00:00:00'), Timestamp('2023-06-08 00:00:00'), Timestamp('2023-11-30 00:00:00')]


Both formats now sort in the correct chronological order. The format codes follow Python's `strftime`/`strptime` conventions: `%B` is the full month name, `%m` is the zero-padded month number, `%d` is the zero-padded day, and `%Y` is the four-digit year.

If you leave off `format=` and let pandas infer the format, it usually works but can be slow on large datasets and occasionally guesses wrong (interpreting `01/02/2023` as January 2nd vs. February 1st depends on locale).

### Mixed formats

Data merged from multiple sources sometimes has a date column where different rows use different formats. Passing `format="mixed"` lets pandas infer the format row by row.

In [5]:
mixed_dates = pd.Series([
    "2023-01-15",
    "01/16/2023",
    "January 17, 2023",
    "2023-01-18"
])

parsed_mixed = pd.to_datetime(mixed_dates, format="mixed")
print(parsed_mixed.tolist())

[Timestamp('2023-01-15 00:00:00'), Timestamp('2023-01-16 00:00:00'), Timestamp('2023-01-17 00:00:00'), Timestamp('2023-01-18 00:00:00')]


All four rows were parsed correctly despite using three different formats. `format="mixed"` is a convenience for messy data, but be aware that it can misinterpret ambiguous dates (like `01/02/2023`, which could be January 2nd or February 1st). If you know the format, specifying it explicitly is safer.

## The `.dt` accessor: extracting date components

Once a column has `datetime64` dtype, the `.dt` accessor gives you direct access to every component of the date. This is the datetime equivalent of the `.str` accessor you learned in notebook 08.4.

To demonstrate `.dt`, we switch to seaborn's `taxis` dataset, which already has its pickup and dropoff columns stored as `datetime64`. That lets us skip the parsing step and go straight to showing what `.dt` can do.

In [6]:
taxis = sns.load_dataset("taxis")
taxis = taxis[["pickup", "dropoff", "passengers", "distance", "fare", "payment"]].copy()
print("Pickup dtype:", taxis["pickup"].dtype)
taxis.head()

Pickup dtype: datetime64[us]


,pickup,dropoff,passengers,distance,fare,payment
0,2019-03-23 20:21:09,2019-03-23 20:27:24,1,1.60,7.0,credit card
1,2019-03-04 16:11:55,2019-03-04 16:19:00,1,0.79,5.0,cash
2,2019-03-27 17:53:01,2019-03-27 18:00:25,1,1.37,7.5,credit card
3,2019-03-10 01:23:59,2019-03-10 01:49:51,1,7.70,27.0,credit card
4,2019-03-30 13:27:42,2019-03-30 13:37:14,3,2.16,9.0,credit card


The taxis dataset has 6,433 rides. The `pickup` and `dropoff` columns already have `datetime64[ns]` dtype; seaborn parsed them on load. Each row shows a ride with a pickup time, dropoff time, passenger count, distance, fare, and payment method. Now use `.dt` to pull individual components out of the pickup column.

In [7]:
taxis["year"]    = taxis["pickup"].dt.year
taxis["month"]   = taxis["pickup"].dt.month
taxis["day"]     = taxis["pickup"].dt.day
taxis["weekday"] = taxis["pickup"].dt.day_name()
taxis["hour"]    = taxis["pickup"].dt.hour

taxis[["pickup", "year", "month", "day", "weekday", "hour"]].head(6)

,pickup,year,month,day,weekday,hour
0,2019-03-23 20:21:09,2019,3,23,Saturday,20
1,2019-03-04 16:11:55,2019,3,4,Monday,16
2,2019-03-27 17:53:01,2019,3,27,Wednesday,17
3,2019-03-10 01:23:59,2019,3,10,Sunday,1
4,2019-03-30 13:27:42,2019,3,30,Saturday,13
5,2019-03-11 10:37:23,2019,3,11,Monday,10


Each `.dt` attribute returns a numeric column (or string for `day_name()`). These extracted columns can now be used directly in groupby aggregations, filters, and plots.

`.dt.dayofweek` gives a number (0 = Monday, 6 = Sunday) instead of a name, which is useful for sorting. `.dt.day_name()` gives the full name, which is better for display.

In [8]:
# Average fare by hour of day
fare_by_hour = (
    taxis.groupby("hour")["fare"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "avg_fare", "count": "n_rides"})
    .round(2)
)
fare_by_hour.head(12)

,avg_fare,n_rides
hour,,
0,14.04,205
1,12.15,110
2,12.07,101
3,12.47,67
4,15.45,57
5,16.77,51
6,13.30,142
7,13.10,221
8,12.22,313


This groupby was only possible after extracting `hour` from the timestamp. Extracting a time component and then grouping by that component is one of the most common operations in any dataset that has timestamps.

### Bucketing timestamps with `.dt.floor()` and `.dt.round()`

Sometimes you want to group by a time window rather than an exact timestamp. `.dt.floor()` truncates to a given precision; `.dt.round()` rounds to the nearest interval.

In [9]:
sample = taxis["pickup"].head(5)
print("Original:        ", sample.tolist())
print("Floored to hour: ", sample.dt.floor("h").tolist())
print("Rounded to hour: ", sample.dt.round("h").tolist())

Original:         [Timestamp('2019-03-23 20:21:09'), Timestamp('2019-03-04 16:11:55'), Timestamp('2019-03-27 17:53:01'), Timestamp('2019-03-10 01:23:59'), Timestamp('2019-03-30 13:27:42')]
Floored to hour:  [Timestamp('2019-03-23 20:00:00'), Timestamp('2019-03-04 16:00:00'), Timestamp('2019-03-27 17:00:00'), Timestamp('2019-03-10 01:00:00'), Timestamp('2019-03-30 13:00:00')]
Rounded to hour:  [Timestamp('2019-03-23 20:00:00'), Timestamp('2019-03-04 16:00:00'), Timestamp('2019-03-27 18:00:00'), Timestamp('2019-03-10 01:00:00'), Timestamp('2019-03-30 13:00:00')]


`floor("h")` always truncates to the start of the current hour. `round("h")` rounds to the nearest hour boundary. Once timestamps are floored, a `groupby` on the floored column groups by hour without needing to extract `hour` separately.

## Timedelta: computing durations

Subtracting two `datetime64` columns gives a `timedelta64` column representing the elapsed time. This is one of the most useful operations in any dataset that records start and end times.

In [10]:
taxis["duration"] = taxis["dropoff"] - taxis["pickup"]
print("dtype:", taxis["duration"].dtype)
taxis[["pickup", "dropoff", "duration"]].head(5)

dtype: timedelta64[us]


,pickup,dropoff,duration
0,2019-03-23 20:21:09,2019-03-23 20:27:24,0 days 00:06:15
1,2019-03-04 16:11:55,2019-03-04 16:19:00,0 days 00:07:05
2,2019-03-27 17:53:01,2019-03-27 18:00:25,0 days 00:07:24
3,2019-03-10 01:23:59,2019-03-10 01:49:51,0 days 00:25:52
4,2019-03-30 13:27:42,2019-03-30 13:37:14,0 days 00:09:32


The subtraction produces a `timedelta64` column that shows the duration of each ride in days, hours, minutes, and seconds. Most rides are a few minutes long, which shows up as `0 days 00:05:00` or similar.

In [11]:
# Convert to minutes (a more natural unit for taxi rides)
taxis["duration_min"] = taxis["duration"].dt.total_seconds() / 60

# Average ride duration by payment type
(
    taxis.groupby("payment")["duration_min"]
    .mean()
    .round(1)
    .sort_values(ascending=False)
)

payment
credit card    15.1
cash           12.6
Name: duration_min, dtype: float64

`.dt.total_seconds()` converts the timedelta to a float in seconds, which you then divide to get minutes. Once duration is a plain numeric column, you can use it in any aggregation, filter, or visualization.

The pattern `(end - start).dt.total_seconds() / 60` is the standard way to compute duration in minutes from two timestamp columns.

## A round-trip: string to datetime and back

When you load a CSV file, date columns arrive as plain strings. The following example makes the two-step sequence concrete: here is what the strings look like, and here is what they become after `pd.to_datetime()`. This is the same process you apply to any real CSV with a timestamp column.

In [ ]:
# Simulate a timestamp column as it arrives in a CSV file
sample_strings = pd.Series([
    "2019-03-23 20:21:09",
    "2019-03-04 16:11:55",
    "2019-03-27 17:53:01"
])
print("As strings (dtype:", sample_strings.dtype, "):")
print(sample_strings.tolist())

# Parse to datetime64 using the matching format string
sample_parsed = pd.to_datetime(sample_strings, format="%Y-%m-%d %H:%M:%S")
print("\nParsed (dtype:", sample_parsed.dtype, "):")
print(sample_parsed.tolist())

The strings look exactly like dates but have `str` dtype, which means they cannot be sorted chronologically, filtered by range, or subtracted. After `pd.to_datetime()` with the matching format string, the dtype is `datetime64` and all of those operations work. The format string `"%Y-%m-%d %H:%M:%S"` maps directly to the structure of the input: `%Y` is the four-digit year, `%m` the month, `%d` the day, `%H` the 24-hour hour, `%M` the minute, and `%S` the second. Identify the format from a sample row, write the format string once, apply it to the whole column.